## Detour reranking

`detour.DetourRanker` reranks a search response by the *excursion distance* of each result,
that is the extra distance the driver pays to stop there:

```
excursion = dist(at → result) + dist(result → stop) − dist(at → stop)
```

Both legs come from a **single** Routing API call per result, using a via waypoint:
`at → via(result) → stop`. The response sections give `dur_to/dist_to/poly_to` and
`dur_from/dist_from/poly_from` at once. One extra call gives the `at → stop` baseline, and
all of them are fired in parallel with `asyncio.gather()`.

Like `RouteEngine`, `DetourRanker` has no widget dependency: it takes a `Response` and
returns a new `Response`.

The cell below rebuilds the route and the search of the [previous notebook](obm_6_route.ipynb):

In [ ]:
from here_search_demo.api import API, HTTPSession
from here_search_demo.auth import Credentials
from here_search_demo.route_engine import RouteEngine

credentials = Credentials()
api = API(credentials=credentials)
session = HTTPSession()
routing_calls = []

engine = RouteEngine(credentials=credentials, on_routing_request=lambda: routing_calls.append(1))
engine.set_route_start((48.57436, 7.71556))  # Strasbourg
engine.set_route_stop((45.75049, 4.84418))  # Lyon
engine.set_route_width(500)
await engine.update_route_attributes()
engine.set_current_position((48.03529, 7.37121))  # near Colmar, already on the way

latitude, longitude = engine.search_at_position
resp = await api.browse(
    session=session,
    latitude=latitude,
    longitude=longitude,
    categories=["100-1000"],
    polyline=engine.search_flexpolyline,
    width=engine.width,
    limit=20,
    lang="en",
)
resp.titles

### One via-route call per result

`DetourRanker.retrieve_detours()` is the low-level entry point: it takes a list of
`(latitude, longitude, street_name_hint)` waypoints and returns the `at → stop` baseline
summary plus one six-tuple per waypoint. The optional street name is passed as the Routing
API `nameHint` waypoint option, which helps the router pick the right side of a complex
intersection.

In [ ]:
from here_search_demo.detour import DetourRanker

ranker = DetourRanker(
    credentials=credentials,
    at_pos=engine.search_at_position,
    stop_pos=engine.stop_position,
    on_routing_request=lambda: routing_calls.append(1),
    route_cache=engine._route_cache,  # share the cache filled by RouteEngine
)

item = resp.data["items"][0]
summary, legs = await ranker.retrieve_detours(
    [(item["position"]["lat"], item["position"]["lng"], item.get("address", {}).get("street"))]
)
dur_to, dist_to, poly_to, dur_from, dist_from, poly_from = legs[0]

print("baseline at → stop:", summary[0], "s,", summary[1], "m")
print("at → item:         ", dur_to, "s,", dist_to, "m")
print("item → stop:       ", dur_from, "s,", dist_from, "m")
print("excursion:         ", dist_to + dist_from - summary[1], "m")

### Reranking a whole response

`rerank()` does that for every positioned item, sorts them by `dur_to` — the time needed
to *reach* the result from the current position — and returns a **new** `Response` object.
The original response is left untouched, so the demo can restore the initial order when
the *travel time* checkbox is switched off.

In [ ]:
calls_before = len(routing_calls)
reranked = await ranker.rerank(resp)
print(
    len(routing_calls) - calls_before,
    "routing requests for",
    len(resp.data["items"]),
    "items (the baseline and the first via-route were already cached)",
)

for item in reranked.data["items"]:
    print(item["_detour"]["label"])

Each reranked item carries a `_detour` annotation. `widgets.state.SearchState.title_for()`
picks up `_detour["label"]` for the result buttons, and `widgets.output_map` reuses the two
polylines to draw the detour without any additional call:

In [ ]:
detour = reranked.data["items"][0]["_detour"]
{key: value for key, value in detour.items() if not key.startswith("polyline")}

In [ ]:
template = "{:<26} {:>9} {:>11} {:>12} {:>12}"
print(template.format("title", "to (min)", "from (min)", "excursion", "round-trip"))
for item in reranked.data["items"]:
    detour = item["_detour"]
    print(
        template.format(
            item["title"][:25],
            round(detour["dur_to_sec"] / 60),
            round(detour["dur_from_sec"] / 60),
            f"{detour['excursion_m']}m",
            round((detour["dur_to_sec"] + detour["dur_from_sec"]) / 60),
        )
    )

### The cache does the heavy lifting

Reranking the same response a second time costs nothing: every `("via", at, item, stop)`
key is already in the shared cache. Toggling the travel-time ranking, or
redrawing a detour when a result is selected, is therefore free.

In [ ]:
calls_before = len(routing_calls)
reranked_again = await ranker.rerank(resp)
print(len(routing_calls) - calls_before, "routing requests")
[item["title"] for item in reranked_again.data["items"]]

### Dropping the results that are too far off

`max_excursion` (in meters) filters out the results whose excursion exceeds a threshold.
A common choice is a small multiple of the corridor width:

In [ ]:
for max_excursion in (None, 10000, 3000, 2000, 1000):
    filtered = await ranker.rerank(resp, max_excursion=max_excursion)
    print(f"max_excursion={str(max_excursion):>5}: {len(filtered.data['items'])} items")

### Adding detour reranking to an existing application

The three pieces above are enough to bolt the feature onto an application that already
sends search requests:

1. Acquire a route once and keep a **shared `route_cache` dict**.
2. Add `polyline` and `width` to the search request (see `API._apply_route()`); this alone
   restricts the results to the road corridor.
3. When the client-side mode is on, `await DetourRanker(...).rerank(resp)` before handing
   the response to the presentation layer, and fall back to the original response on error
   — that is exactly what `widgets.app.OneBoxMap._handle_result_list_with_detour()` does.
4. Read `_detour["label"]` for the titles and `_detour["polyline_to"] / ["polyline_from"]`
   for the map, as shown in the [next notebook](obm_8_detour_map.ipynb).

Because reranking is asynchronous, run it in a task and keep the un-reranked response
around so the user can toggle the mode without a new search request.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>